# Game Recomendations on Steam Platform 

Libraries

In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
import ast
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

DataFiles

In [2]:
# Read the data from files
path = 'data'
df_games = pd.read_csv(path+"/games.csv")
df_users = pd.read_csv(path+"/users.csv")
df_recommendations = pd.read_csv(path+"/recommendations.csv")
df_genre = pd.read_json(path+'/games_metadata.json', lines=True)

Overview

In [3]:
df_games.head(10)

,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
0,13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,0.0,True
1,22364,BRINK: Agents of Change,2011-08-03,True,False,False,Positive,85,21,2.99,2.99,0.0,True
2,113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,0.0,True
3,226560,Escape Dead Island,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,0.0,True
4,249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,0.0,True
5,250180,METAL SLUG 3,2015-09-14,True,False,False,Very Positive,90,5579,7.99,7.99,0.0,True
6,253980,Enclave,2013-10-04,True,True,True,Mostly Positive,75,1608,4.99,4.99,0.0,True
7,271850,Men of War: Assault Squad 2 - Deluxe Edition u...,2014-05-16,True,False,False,Mixed,61,199,6.99,6.99,0.0,True
8,282900,Hyperdimension Neptunia Re;Birth1,2015-01-29,True,False,False,Very Positive,94,9686,14.99,14.99,0.0,True
9,19810,The Sum of All Fears,2008-10-10,True,False,False,Mostly Positive,75,33,9.99,9.99,0.0,True


In [4]:
df_users.head(10)

,user_id,products,reviews
0,7360263,359,0
1,14020781,156,1
2,8762579,329,4
3,4820647,176,4
4,5167327,98,2
5,5664667,145,5
6,5889167,447,2
7,7281762,1083,1
8,7445952,273,1
9,7462927,51,1


In [5]:
df_recommendations.head(10)

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,975370,0,0,2022-12-12,True,36.3,51580,0
1,304390,4,0,2017-02-17,False,11.5,2586,1
2,1085660,2,0,2019-11-17,True,336.5,253880,2
3,703080,0,0,2022-09-23,True,27.4,259432,3
4,526870,0,0,2021-01-10,True,7.9,23869,4
5,306130,0,0,2021-10-10,True,8.6,45425,5
6,238960,0,0,2017-11-25,True,538.8,88282,6
7,730,0,0,2021-11-30,False,157.5,63209,7
8,255710,0,0,2021-05-21,True,18.7,354512,8
9,289070,0,0,2020-05-26,True,397.5,454422,9


In [6]:
df_genre.head(10)

,app_id,description,tags
0,13500,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
1,22364,,[Action]
2,113020,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
3,226560,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
4,249050,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."
5,250180,"“METAL SLUG 3”, the masterpiece in SNK’s emble...","[Arcade, Classic, Action, Co-op, Side Scroller..."
6,253980,Experience incredibly atmospheric and intense ...,"[RPG, Action, Fantasy, Third Person, Hack and ..."
7,271850,,"[Strategy, Simulation, Action, RTS, World War II]"
8,282900,"Packed with fast-paced, turn-based RPG action,...","[Anime, JRPG, Female Protagonist, Cute, RPG, S..."
9,19810,Lead a team of domestic counter-terrorism expe...,"[Action, Tactical]"


In [7]:
#Metadata of Dataframes
df_list = [
    ("df_games", df_games),
    ("df_users", df_users),
    ("df_recommendations", df_recommendations),
    ("df_genre", df_genre)
]

for name, df in df_list:
    print(f"\n📄 DataFrame: {name}")
    df.info()


📄 DataFrame: df_games
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50872 entries, 0 to 50871
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   app_id          50872 non-null  int64  
 1   title           50872 non-null  object 
 2   date_release    50872 non-null  object 
 3   win             50872 non-null  bool   
 4   mac             50872 non-null  bool   
 5   linux           50872 non-null  bool   
 6   rating          50872 non-null  object 
 7   positive_ratio  50872 non-null  int64  
 8   user_reviews    50872 non-null  int64  
 9   price_final     50872 non-null  float64
 10  price_original  50872 non-null  float64
 11  discount        50872 non-null  float64
 12  steam_deck      50872 non-null  bool   
dtypes: bool(4), float64(3), int64(3), object(3)
memory usage: 3.7+ MB

📄 DataFrame: df_users
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14306064 entries, 0 to 14306063
Data columns (tot

In [8]:
# Converting all object columns to string dtype
df_games = df_games.astype({col: 'string' for col in df_games.select_dtypes(include='object').columns})
df_genre = df_genre.astype({col: 'string' for col in df_genre.select_dtypes(include='object').columns})

# Converting the date related columns to date dtype
df_games['date_release'] = pd.to_datetime(df_games['date_release'])
df_recommendations['date'] = pd.to_datetime(df_recommendations['date'])

In [9]:
#Metadata of Dataframes after datatype conversions
df_list = [
    ("df_games", df_games),
    ("df_users", df_users),
    ("df_recommendations", df_recommendations),
    ("df_genre", df_genre)
]

for name, df in df_list:
    print(f"\n📄 DataFrame: {name}")
    df.info()


📄 DataFrame: df_games
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50872 entries, 0 to 50871
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   app_id          50872 non-null  int64         
 1   title           50872 non-null  string        
 2   date_release    50872 non-null  datetime64[ns]
 3   win             50872 non-null  bool          
 4   mac             50872 non-null  bool          
 5   linux           50872 non-null  bool          
 6   rating          50872 non-null  string        
 7   positive_ratio  50872 non-null  int64         
 8   user_reviews    50872 non-null  int64         
 9   price_final     50872 non-null  float64       
 10  price_original  50872 non-null  float64       
 11  discount        50872 non-null  float64       
 12  steam_deck      50872 non-null  bool          
dtypes: bool(4), datetime64[ns](1), float64(3), int64(3), string(2)
memory usage: 3.7 MB

In [10]:
for name, df in df_list:
    print(f"Total No. of records in \n📄 DataFrame: {name}")
    print(df.shape[0])

Total No. of records in 
📄 DataFrame: df_games
50872
Total No. of records in 
📄 DataFrame: df_users
14306064
Total No. of records in 
📄 DataFrame: df_recommendations
41154794
Total No. of records in 
📄 DataFrame: df_genre
50872


In [11]:
#Checking for Null Values
for name, df in df_list:
    print(f"\n📄 DataFrame Null Values: {name}")
    print(df.isna().sum())


📄 DataFrame Null Values: df_games
app_id            0
title             0
date_release      0
win               0
mac               0
linux             0
rating            0
positive_ratio    0
user_reviews      0
price_final       0
price_original    0
discount          0
steam_deck        0
dtype: int64

📄 DataFrame Null Values: df_users
user_id     0
products    0
reviews     0
dtype: int64

📄 DataFrame Null Values: df_recommendations
app_id            0
helpful           0
funny             0
date              0
is_recommended    0
hours             0
user_id           0
review_id         0
dtype: int64

📄 DataFrame Null Values: df_genre
app_id         0
description    0
tags           0
dtype: int64


In [12]:
#Checking for 'app_id' Duplicate Values in all dataframes
for name, df in df_list:
    if 'app_id' in df:
        print(f'Total records count in {name} 📄: ',df.shape[0])
        print(f'Distinct app_id records in {name} 📄: ',df['app_id'].nunique())
        
#Checking for 'user_id' Duplicate Values in df_users
print(f'Total records count in df_users 📄: ',df_users.shape[0])
print(f'Distinct user_id records in df_users 📄: ',df_users['user_id'].nunique()) 

#Checking for duplicated based on combination of app_id & their description
print(f'Total duplicate records count in df_games 📄: ',df_games.duplicated(subset=['app_id', 'title']).sum())
print(f'Total duplicate records count in df_genre 📄: ',df_genre.duplicated(subset=['app_id','description']).sum())     
        

Total records count in df_games 📄:  50872
Distinct app_id records in df_games 📄:  50872
Total records count in df_recommendations 📄:  41154794
Distinct app_id records in df_recommendations 📄:  37610
Total records count in df_genre 📄:  50872
Distinct app_id records in df_genre 📄:  50872
Total records count in df_users 📄:  14306064
Distinct user_id records in df_users 📄:  14306064
Total duplicate records count in df_games 📄:  0
Total duplicate records count in df_genre 📄:  0


Data Preprocessing

In [13]:
#Focusing only on the games available on macos
counts = df_games['mac'].value_counts()
print(counts)

mac
False    37854
True     13018
Name: count, dtype: int64


In [14]:
df_games_processed = df_games.copy()
df_games_processed = df_games_processed[df_games_processed['mac'] == True]

In [15]:
df_games_processed

,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
2,113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,0.0,True
4,249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,0.0,True
6,253980,Enclave,2013-10-04,True,True,True,Mostly Positive,75,1608,4.99,4.99,0.0,True
13,29180,Osmos,2009-08-18,True,True,True,Very Positive,88,532,9.99,9.99,0.0,True
18,245950,Borderlands 2: Headhunter 4: Wedding Day Massacre,2014-02-11,True,True,True,Very Positive,84,294,0.89,2.99,70.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50830,2195430,Two Point Campus: Medical School,2023-08-17,True,True,True,Mixed,50,22,10.00,0.00,0.0,True
50835,2446110,Stacklands: Cursed Worlds,2023-07-25,True,True,False,Very Positive,90,62,4.00,0.00,0.0,True
50848,2515460,Northgard - Kernev Clan of the Stoat,2023-08-24,True,True,True,Mixed,67,80,5.00,0.00,0.0,True
50851,1555150,Pocket Bravery,2023-08-31,True,True,True,Very Positive,89,248,20.00,0.00,0.0,True


In [16]:
df_games_processed.drop(columns=['win', 'mac','linux','steam_deck','positive_ratio', 'discount', 'price_original'], inplace=True)

In [17]:
# Define custom order from worst to best (if reversed)
order = ["Overwhelmingly Positive", "Very Positive", "Positive", "Mostly Positive",
         "Mixed", "Mostly Negative", "Negative", "Very Negative", "Overwhelmingly Negative"][::-1]

# Initialize encoder with the custom order
enc = OrdinalEncoder(categories=[order])

# Transform the 'rating' column to ordinal values
df_games_processed[['rating']] = enc.fit_transform(df_games_processed[['rating']])

In [18]:
df_games_processed

,app_id,title,date_release,rating,user_reviews,price_final
2,113020,Monaco: What's Yours Is Mine,2013-04-24,7.0,3722,14.99
4,249050,Dungeon of the ENDLESS™,2014-10-27,7.0,8784,11.99
6,253980,Enclave,2013-10-04,5.0,1608,4.99
13,29180,Osmos,2009-08-18,7.0,532,9.99
18,245950,Borderlands 2: Headhunter 4: Wedding Day Massacre,2014-02-11,7.0,294,0.89
...,...,...,...,...,...,...
50830,2195430,Two Point Campus: Medical School,2023-08-17,4.0,22,10.00
50835,2446110,Stacklands: Cursed Worlds,2023-07-25,7.0,62,4.00
50848,2515460,Northgard - Kernev Clan of the Stoat,2023-08-24,4.0,80,5.00
50851,1555150,Pocket Bravery,2023-08-31,7.0,248,20.00


In [19]:
games_processed = pd.merge(df_games_processed, df_genre[['app_id', 'tags']], on='app_id', how='inner')

In [20]:
games_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13018 entries, 0 to 13017
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   app_id        13018 non-null  int64         
 1   title         13018 non-null  string        
 2   date_release  13018 non-null  datetime64[ns]
 3   rating        13018 non-null  float64       
 4   user_reviews  13018 non-null  int64         
 5   price_final   13018 non-null  float64       
 6   tags          13018 non-null  string        
dtypes: datetime64[ns](1), float64(2), int64(2), string(2)
memory usage: 712.1 KB


In [21]:
games_processed

,app_id,title,date_release,rating,user_reviews,price_final,tags
0,113020,Monaco: What's Yours Is Mine,2013-04-24,7.0,3722,14.99,"['Co-op', 'Stealth', 'Indie', 'Heist', 'Local ..."
1,249050,Dungeon of the ENDLESS™,2014-10-27,7.0,8784,11.99,"['Roguelike', 'Strategy', 'Tower Defense', 'Pi..."
2,253980,Enclave,2013-10-04,5.0,1608,4.99,"['RPG', 'Action', 'Fantasy', 'Third Person', '..."
3,29180,Osmos,2009-08-18,7.0,532,9.99,"['Indie', 'Casual', 'Puzzle', 'Relaxing', 'Sin..."
4,245950,Borderlands 2: Headhunter 4: Wedding Day Massacre,2014-02-11,7.0,294,0.89,"['Action', 'RPG', 'FPS', 'Co-op', 'Shooter', '..."
...,...,...,...,...,...,...,...
13013,2195430,Two Point Campus: Medical School,2023-08-17,4.0,22,10.00,[]
13014,2446110,Stacklands: Cursed Worlds,2023-07-25,7.0,62,4.00,"['Simulation', 'Indie', 'Casual', 'Survival', ..."
13015,2515460,Northgard - Kernev Clan of the Stoat,2023-08-24,4.0,80,5.00,"['Strategy', 'Indie', 'Simulation']"
13016,1555150,Pocket Bravery,2023-08-31,7.0,248,20.00,[]


In [22]:
games_processed = games_processed[games_processed['tags'].apply(lambda x: len(x) > 2)]

In [23]:
games_processed

,app_id,title,date_release,rating,user_reviews,price_final,tags
0,113020,Monaco: What's Yours Is Mine,2013-04-24,7.0,3722,14.99,"['Co-op', 'Stealth', 'Indie', 'Heist', 'Local ..."
1,249050,Dungeon of the ENDLESS™,2014-10-27,7.0,8784,11.99,"['Roguelike', 'Strategy', 'Tower Defense', 'Pi..."
2,253980,Enclave,2013-10-04,5.0,1608,4.99,"['RPG', 'Action', 'Fantasy', 'Third Person', '..."
3,29180,Osmos,2009-08-18,7.0,532,9.99,"['Indie', 'Casual', 'Puzzle', 'Relaxing', 'Sin..."
4,245950,Borderlands 2: Headhunter 4: Wedding Day Massacre,2014-02-11,7.0,294,0.89,"['Action', 'RPG', 'FPS', 'Co-op', 'Shooter', '..."
...,...,...,...,...,...,...,...
13004,1410330,Love Shore,2023-06-30,4.0,36,14.99,"['Cyberpunk', 'RPG', 'Choices Matter', 'Noir',..."
13005,2159650,Drift,2023-05-12,4.0,36,15.99,"['Open World Survival Craft', 'Survival', 'Onl..."
13011,2349040,Dinky Guardians,2023-10-02,6.0,16,13.00,"['Automation', 'Simulation', 'Co-op', 'Colony ..."
13014,2446110,Stacklands: Cursed Worlds,2023-07-25,7.0,62,4.00,"['Simulation', 'Indie', 'Casual', 'Survival', ..."


In [24]:
#Converting tags to string
games_processed['tags'] = games_processed['tags'].apply(ast.literal_eval)
games_processed['tags'] = games_processed['tags'].apply(lambda tags: ' '.join(tags))
games_processed['tags'] = games_processed['tags'].astype('string')


In [25]:
games_processed

,app_id,title,date_release,rating,user_reviews,price_final,tags
0,113020,Monaco: What's Yours Is Mine,2013-04-24,7.0,3722,14.99,Co-op Stealth Indie Heist Local Co-Op Strategy...
1,249050,Dungeon of the ENDLESS™,2014-10-27,7.0,8784,11.99,Roguelike Strategy Tower Defense Pixel Graphic...
2,253980,Enclave,2013-10-04,5.0,1608,4.99,RPG Action Fantasy Third Person Hack and Slash...
3,29180,Osmos,2009-08-18,7.0,532,9.99,Indie Casual Puzzle Relaxing Singleplayer Phys...
4,245950,Borderlands 2: Headhunter 4: Wedding Day Massacre,2014-02-11,7.0,294,0.89,Action RPG FPS Co-op Shooter Action RPG Online...
...,...,...,...,...,...,...,...
13004,1410330,Love Shore,2023-06-30,4.0,36,14.99,Cyberpunk RPG Choices Matter Noir Interactive ...
13005,2159650,Drift,2023-05-12,4.0,36,15.99,Open World Survival Craft Survival Online Co-O...
13011,2349040,Dinky Guardians,2023-10-02,6.0,16,13.00,Automation Simulation Co-op Colony Sim Craftin...
13014,2446110,Stacklands: Cursed Worlds,2023-07-25,7.0,62,4.00,Simulation Indie Casual Survival Card Game Sol...


In [26]:
games_processed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12737 entries, 0 to 13015
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   app_id        12737 non-null  int64         
 1   title         12737 non-null  string        
 2   date_release  12737 non-null  datetime64[ns]
 3   rating        12737 non-null  float64       
 4   user_reviews  12737 non-null  int64         
 5   price_final   12737 non-null  float64       
 6   tags          12737 non-null  string        
dtypes: datetime64[ns](1), float64(2), int64(2), string(2)
memory usage: 796.1 KB


In [27]:
# Choose the numeric columns to normalize
numeric_cols = ['rating', 'user_reviews', 'price_final']

# Fit scaler and transform
scaler = MinMaxScaler()
games_processed[numeric_cols] = scaler.fit_transform(games_processed[numeric_cols])


In [28]:
games_processed

,app_id,title,date_release,rating,user_reviews,price_final,tags
0,113020,Monaco: What's Yours Is Mine,2013-04-24,0.875,0.007208,0.055521,Co-op Stealth Indie Heist Local Co-Op Strategy...
1,249050,Dungeon of the ENDLESS™,2014-10-27,0.875,0.017037,0.044409,Roguelike Strategy Tower Defense Pixel Graphic...
2,253980,Enclave,2013-10-04,0.625,0.003103,0.018482,RPG Action Fantasy Third Person Hack and Slash...
3,29180,Osmos,2009-08-18,0.875,0.001014,0.037001,Indie Casual Puzzle Relaxing Singleplayer Phys...
4,245950,Borderlands 2: Headhunter 4: Wedding Day Massacre,2014-02-11,0.875,0.000551,0.003296,Action RPG FPS Co-op Shooter Action RPG Online...
...,...,...,...,...,...,...,...
13004,1410330,Love Shore,2023-06-30,0.500,0.000050,0.055521,Cyberpunk RPG Choices Matter Noir Interactive ...
13005,2159650,Drift,2023-05-12,0.500,0.000050,0.059224,Open World Survival Craft Survival Online Co-O...
13011,2349040,Dinky Guardians,2023-10-02,0.750,0.000012,0.048150,Automation Simulation Co-op Colony Sim Craftin...
13014,2446110,Stacklands: Cursed Worlds,2023-07-25,0.875,0.000101,0.014815,Simulation Indie Casual Survival Card Game Sol...


In [29]:
sample_games = games_processed[games_processed['date_release'].dt.year >= 2020]

In [30]:
sample_games.reset_index(drop=True, inplace=True)
sample_games

,app_id,title,date_release,rating,user_reviews,price_final,tags
0,1872790,Luckitown,2022-01-19,0.875,0.000297,0.018482,Simulation Tower Defense Strategy Turn-Based S...
1,1259750,Tropico 6 - Spitter,2020-04-23,0.500,0.000050,0.037001,Simulation Strategy
2,498310,RPG in a Box,2022-05-10,0.875,0.000278,0.111078,Game Development Animation & Modeling Design &...
3,672630,Academia : School Simulator,2021-01-28,0.875,0.004194,0.074040,Management City Builder Base Building Colony S...
4,914890,Nine Noir Lives,2022-09-07,0.750,0.000056,0.074040,Point & Click Comedy 2D Noir Cute Detective Fu...
...,...,...,...,...,...,...,...
4141,1410330,Love Shore,2023-06-30,0.500,0.000050,0.055521,Cyberpunk RPG Choices Matter Noir Interactive ...
4142,2159650,Drift,2023-05-12,0.500,0.000050,0.059224,Open World Survival Craft Survival Online Co-O...
4143,2349040,Dinky Guardians,2023-10-02,0.750,0.000012,0.048150,Automation Simulation Co-op Colony Sim Craftin...
4144,2446110,Stacklands: Cursed Worlds,2023-07-25,0.875,0.000101,0.014815,Simulation Indie Casual Survival Card Game Sol...


In [31]:
common_game_ids = sample_games['app_id'].unique()   # all game IDs in df2
recommendations_df = df_recommendations[df_recommendations['app_id'].isin(common_game_ids)].copy()

In [32]:
recommendations_df

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
73932,1218210,8,0,2022-07-13,True,119.5,4159892,73932
76525,1218210,6,0,2022-04-09,True,24.3,9824132,76525
132577,1218210,14,0,2022-04-01,True,11.7,11311605,132577
254185,1218210,2,0,2022-06-30,True,32.7,3830708,254185
300086,1218210,123,36,2022-04-04,False,32.5,11545696,300086
...,...,...,...,...,...,...,...,...
41154745,1104450,0,0,2020-07-29,True,3.0,9985656,41154745
41154746,893850,0,0,2020-09-14,True,128.0,821144,41154746
41154748,1404850,0,0,2021-04-15,True,8.0,11065427,41154748
41154769,1138660,0,0,2021-08-13,False,11.0,10433531,41154769


In [33]:
recommendations_df.count()

app_id            1279263
helpful           1279263
funny             1279263
date              1279263
is_recommended    1279263
hours             1279263
user_id           1279263
review_id         1279263
dtype: int64

In [34]:
#Droping the review_id column & coverting booolean to numeric
recommendations_df.drop(columns=['review_id'], inplace=True)
recommendations_df['is_recommended'] = recommendations_df['is_recommended'].astype(int)

In [35]:
recommendations_df

,app_id,helpful,funny,date,is_recommended,hours,user_id
73932,1218210,8,0,2022-07-13,1,119.5,4159892
76525,1218210,6,0,2022-04-09,1,24.3,9824132
132577,1218210,14,0,2022-04-01,1,11.7,11311605
254185,1218210,2,0,2022-06-30,1,32.7,3830708
300086,1218210,123,36,2022-04-04,0,32.5,11545696
...,...,...,...,...,...,...,...
41154745,1104450,0,0,2020-07-29,1,3.0,9985656
41154746,893850,0,0,2020-09-14,1,128.0,821144
41154748,1404850,0,0,2021-04-15,1,8.0,11065427
41154769,1138660,0,0,2021-08-13,0,11.0,10433531


In [36]:
count = (recommendations_df['hours'] < 2).sum()
count

222962

In [37]:
recommendations_df = recommendations_df[recommendations_df['hours'] > 2].copy()

In [38]:
recommendations_df

,app_id,helpful,funny,date,is_recommended,hours,user_id
73932,1218210,8,0,2022-07-13,1,119.5,4159892
76525,1218210,6,0,2022-04-09,1,24.3,9824132
132577,1218210,14,0,2022-04-01,1,11.7,11311605
254185,1218210,2,0,2022-06-30,1,32.7,3830708
300086,1218210,123,36,2022-04-04,0,32.5,11545696
...,...,...,...,...,...,...,...
41154745,1104450,0,0,2020-07-29,1,3.0,9985656
41154746,893850,0,0,2020-09-14,1,128.0,821144
41154748,1404850,0,0,2021-04-15,1,8.0,11065427
41154769,1138660,0,0,2021-08-13,0,11.0,10433531


In [39]:
#Compute max hours_played per user
user_max_hours = recommendations_df.groupby('user_id')['hours'].max().reset_index()
user_max_hours.rename(columns={'hours': 'max_hours'}, inplace=True)

# Merge with original DataFrame
recommendations_df = recommendations_df.merge(user_max_hours, on='user_id')

#Compute implicit rating
recommendations_df['rating'] = recommendations_df['hours'] / recommendations_df['max_hours']

In [40]:
recommendations_df

,app_id,helpful,funny,date,is_recommended,hours,user_id,max_hours,rating
0,1218210,8,0,2022-07-13,1,119.5,4159892,119.5,1.000000
1,1218210,6,0,2022-04-09,1,24.3,9824132,32.7,0.743119
2,1218210,14,0,2022-04-01,1,11.7,11311605,33.2,0.352410
3,1218210,2,0,2022-06-30,1,32.7,3830708,32.7,1.000000
4,1218210,123,36,2022-04-04,0,32.5,11545696,35.6,0.912921
...,...,...,...,...,...,...,...,...,...
1034565,1104450,0,0,2020-07-29,1,3.0,9985656,3.0,1.000000
1034566,893850,0,0,2020-09-14,1,128.0,821144,128.0,1.000000
1034567,1404850,0,0,2021-04-15,1,8.0,11065427,8.0,1.000000
1034568,1138660,0,0,2021-08-13,0,11.0,10433531,25.0,0.440000


In [41]:
recommendations_df.drop(columns=['max_hours'], inplace=True)

In [42]:
recommendations_df

,app_id,helpful,funny,date,is_recommended,hours,user_id,rating
0,1218210,8,0,2022-07-13,1,119.5,4159892,1.000000
1,1218210,6,0,2022-04-09,1,24.3,9824132,0.743119
2,1218210,14,0,2022-04-01,1,11.7,11311605,0.352410
3,1218210,2,0,2022-06-30,1,32.7,3830708,1.000000
4,1218210,123,36,2022-04-04,0,32.5,11545696,0.912921
...,...,...,...,...,...,...,...,...
1034565,1104450,0,0,2020-07-29,1,3.0,9985656,1.000000
1034566,893850,0,0,2020-09-14,1,128.0,821144,1.000000
1034567,1404850,0,0,2021-04-15,1,8.0,11065427,1.000000
1034568,1138660,0,0,2021-08-13,0,11.0,10433531,0.440000


In [43]:
# Choose the numeric columns to normalize
numeric_cols = ['helpful', 'funny', 'hours']

# Fit scaler and transform
scaler = MinMaxScaler()
recommendations_df[numeric_cols] = scaler.fit_transform(recommendations_df[numeric_cols])

In [44]:
recommendations_df.reset_index(drop=True, inplace=True)
recommendations_df

,app_id,helpful,funny,date,is_recommended,hours,user_id,rating
0,1218210,0.000410,0.000000,2022-07-13,1,0.117647,4159892,1.000000
1,1218210,0.000307,0.000000,2022-04-09,1,0.022247,9824132,0.743119
2,1218210,0.000717,0.000000,2022-04-01,1,0.009620,11311605,0.352410
3,1218210,0.000102,0.000000,2022-06-30,1,0.030664,3830708,1.000000
4,1218210,0.006297,0.004308,2022-04-04,0,0.030464,11545696,0.912921
...,...,...,...,...,...,...,...,...
1034565,1104450,0.000000,0.000000,2020-07-29,1,0.000902,9985656,1.000000
1034566,893850,0.000000,0.000000,2020-09-14,1,0.126165,821144,1.000000
1034567,1404850,0.000000,0.000000,2021-04-15,1,0.005912,11065427,1.000000
1034568,1138660,0.000000,0.000000,2021-08-13,0,0.008919,10433531,0.440000


In [45]:
df_users

,user_id,products,reviews
0,7360263,359,0
1,14020781,156,1
2,8762579,329,4
3,4820647,176,4
4,5167327,98,2
...,...,...,...
14306059,5047430,6,0
14306060,5048153,0,0
14306061,5059205,31,0
14306062,5074363,0,0


In [46]:
# Choose the numeric columns to normalize
numeric_cols = ['products', 'reviews']

# Fit scaler and transform
scaler = MinMaxScaler()
df_users[numeric_cols] = scaler.fit_transform(df_users[numeric_cols])

In [47]:
df_users

,user_id,products,reviews
0,7360263,0.011144,0.000000
1,14020781,0.004843,0.000165
2,8762579,0.010213,0.000662
3,4820647,0.005463,0.000662
4,5167327,0.003042,0.000331
...,...,...,...
14306059,5047430,0.000186,0.000000
14306060,5048153,0.000000,0.000000
14306061,5059205,0.000962,0.000000
14306062,5074363,0.000000,0.000000


In [48]:
common_user_ids = recommendations_df['user_id'].unique()   # all game IDs in df2
users = df_users[df_users['user_id'].isin(common_user_ids)].copy()

In [49]:
users

,user_id,products,reviews
45,4616950,0.021792,0.005955
50,4998568,0.005898,0.000165
59,5847428,0.005650,0.000331
92,9021133,0.038089,0.003639
114,11316351,0.021016,0.011249
...,...,...,...
14305785,4772598,0.000528,0.000165
14305819,4806514,0.000000,0.000165
14305849,4830136,0.000000,0.000165
14305886,4846067,0.028342,0.001820


In [50]:
recommendations_df.to_pickle('recommendations_processed.pkl')
sample_games.to_pickle('games_processed.pkl')
print('DataFrames have been saved successfully!')

DataFrames have been saved successfully!
